In [1]:
from prettytable import PrettyTable

def count_parameters(model):
    table = PrettyTable(["Modules", "Parameters"])
    total_params = 0
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        params = parameter.numel()
        table.add_row([name, params])
        total_params += params
    print(table)
    print(f"Total Trainable Params: {total_params}")
    return total_params


def make_pos_encoding(blk,size): 
    
    blk_x1,blk_x2,blk_y1,blk_y2,blk_z1,blk_z2 = blk
    blk_x2 = blk_x1 + size[0]-1
    blk_y2 = blk_y1 + size[1]-1
    blk_z2 = blk_z1 + size[2]-1
    res = []
    t = []
    for i in range(blk_x1,blk_x2+1):
        l = []
        for j in range(blk_y1,blk_y2+1):
            q = []
            for k in range(blk_z1,blk_z2+1):
                q.append((i,j,k))   
            l.append(q)
        t.append(l)
    res.append(t)
    res = torch.from_numpy(np.asarray(res))
    res = torch.permute(res, (0,4,1,2,3))
    res = res.squeeze()
    return res

def blk_from_coor(coor):
    blk = []
    for i in range(len(rel_coor)):
        blk.append(int(rel_coor[i].min().item()))
        blk.append(int(rel_coor[i].max().item()))
    return blk

def interpolate(inp,size):
    inp = torch.from_numpy(inp)
    inp = torch.permute(inp, (3,0,1,2))
    inp = torch.unsqueeze(inp, 0)
    interpolated = torch.nn.functional.interpolate(inp,size = torch.Size(size))
    interpolated = torch.permute(interpolated, (2,3,4,1,0))
    return interpolated.squeeze()

def align(lr,hr,pred):
    lr,hr,pred = lr.cpu().detach().numpy(),hr.cpu().detach().numpy(),pred.cpu().detach().numpy()
    pos = lr.shape.index(min(lr.shape[:3]))
    if(pos == 1):
        lr = np.transpose(lr,(0,2,1,3))
        hr = np.transpose(hr,(0,2,1,3))
        pred = np.transpose(pred,(0,2,1,3))
    elif(pos == 0):
        lr = np.transpose(lr,(1,2,0,3))
        hr = np.transpose(hr,(1,2,0,3))
        pred = np.transpose(pred,(1,2,0,3))
    return lr,hr,pred
    

In [2]:
import model
import os
from option import args
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import data
import utils
import numpy as np
import json
import h5py

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


number of common Subjects  171


In [3]:
args.no_vols = 2
args.test_vols = 2
# args.tv_en = True
np.random.seed(args.seed)
ids = utils.get_ids()
# ids.sort()
total_vols = args.no_vols+args.test_vols
ids.sort()
ids = ids[:total_vols]
ids = np.random.choice(ids,total_vols,replace = False)


In [4]:
load_dir = "working_models/good_models"

In [5]:
paths = {}
max_ep = 0
for i in os.listdir(load_dir):
    if "check" not in i and "tensor" not in i:
        temp = []
#         print(os.listdir(load_dir + "/" + i))
        for j in os.listdir(load_dir + "/" + i + "/model"):
            temp.append(load_dir + "/" + i + "/model/" + j)
            
            
            _,ep,hfen,psnr = j.split("_")
            ep = int(ep)
            if(max_ep < ep):
                latest_ep_path = load_dir + "/" + i + "/model/" + j
                max_ep = ep
                
            conf = load_dir + "/" + i + "/config.txt"
        paths[i] = (temp,conf,latest_ep_path) 
        

In [6]:
list_models = list(paths.keys())

In [7]:
list_models

['dmri_rdn,3d,train_80,test_20,C,attn_False,growth64,loss_1*L1,bs_8,tvFalse',
 'dmri_rdn,3d,train_80,test_20,C,attn_False,growth32,loss_1*L1,bs_8,tvFalse']

In [8]:
load_model = list_models[0]

In [9]:
with open(paths[load_model][1], 'r') as f:
    args.__dict__ = json.load(f)

In [10]:
args

Namespace(block_size=[48, 48, 6], var_blk_size=False, start_var=True, epochs=100, dir='/storage', batch_size=8, sort=True, debug=False, preload=True, ret_points=False, enable_thres=True, thres=0.7, psnr_sim=22, rel_coord=False, patience=3, no_vols=80, test_vols=20, lr=0.0005, max_lr=0.01, lr_decay=20, decay_type='step', gamma=0.8, optimizer='ADAM', momentum=0.9, beta1=0.9, beta2=0.999, epsilon=1e-08, weight_decay=0, start_epoch=0, loss='1*L1', skip_threshold=100.0, run_name='dmri_rdn,3d,train_80,test_20,C,attn_False,growth64,loss_1*L1,bs_8,tvFalse', save='DTIArb', load='.', save_models=False, resume=0, print_every=20, save_every=30, cpu=False, gpu=0, seed=1, reset=False, pin_mem=False, model='dmri_rdn', in_chans=7, encoder='rdb', tv=False, attention=False, drop_prob=0, growth=64, type='3d', out_chans=5, RDNconfig='C', precision='single', cuda=True, scale=[1, 1, 1], offset=3, stable_epoch=1)

In [11]:
DTI_SR = model.Model(args)
DTI_SR.load(paths[load_model][2])

Making model... here
load from model_working_models/good_models/dmri_rdn,3d,train_80,test_20,C,attn_False,growth64,loss_1*L1,bs_8,tvFalse/model/model_85_0.2709_23.0297.pt.pt


In [12]:
count_parameters(DTI_SR)

+----------------------------------------------+------------+
|                   Modules                    | Parameters |
+----------------------------------------------+------------+
|         model.encoder.SFENet1.weight         |   12096    |
|          model.encoder.SFENet1.bias          |     64     |
|         model.encoder.SFENet2.weight         |   110592   |
|          model.encoder.SFENet2.bias          |     64     |
| model.encoder.RDBs.0.convs.0.layers.0.weight |   55296    |
|  model.encoder.RDBs.0.convs.0.layers.0.bias  |     32     |
| model.encoder.RDBs.0.convs.0.layers.1.weight |     32     |
|  model.encoder.RDBs.0.convs.0.layers.1.bias  |     32     |
| model.encoder.RDBs.0.convs.1.layers.0.weight |   82944    |
|  model.encoder.RDBs.0.convs.1.layers.0.bias  |     32     |
| model.encoder.RDBs.0.convs.1.layers.1.weight |     32     |
|  model.encoder.RDBs.0.convs.1.layers.1.bias  |     32     |
| model.encoder.RDBs.0.convs.2.layers.0.weight |   110592   |
|  model

9367993

In [13]:
ids = utils.get_ids()
ids.sort()
offset = 100
args.no_vols = 1
args.test_vols = 20
total_vols = args.no_vols+args.test_vols
temp = ids[offset:offset+args.no_vols]
temp.extend(ids[offset:args.test_vols+offset])
ids = temp
args.batch_size = 1
print(ids)

['352738', '352738', '360030', '365343', '380036', '381038', '385046', '389357', '393247', '395756', '397760', '401422', '406836', '412528', '429040', '436845', '463040', '467351', '525541', '541943', '547046']


In [ ]:
loader = data.Data(args,ids= ids)

In [ ]:
len(loader.testing_dataset)

In [ ]:
import utility
import skimage.metrics as metrics

In [ ]:
lr_hr = []
pred_hr = []

In [5]:
from ArSSR.model import ArSSR

In [7]:
from model.MetaSR import metasr_rdn

In [8]:
model = metasr_rdn()

In [9]:
model

MetaSR_RDN(
  (conv1): Conv3d(7, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv3d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (trunk): Sequential(
    (0): _ResidualDenseBlock(
      (rdb): Sequential(
        (0): _ResidualBlock(
          (rb): Sequential(
            (0): Conv3d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (1): ReLU(inplace=True)
          )
        )
        (1): _ResidualBlock(
          (rb): Sequential(
            (0): Conv3d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (1): ReLU(inplace=True)
          )
        )
        (2): _ResidualBlock(
          (rb): Sequential(
            (0): Conv3d(192, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (1): ReLU(inplace=True)
          )
        )
        (3): _ResidualBlock(
          (rb): Sequential(
            (0): Conv3d(256, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (1): ReL

In [11]:
count_parameters(model)

+--------------------------------------+------------+
|               Modules                | Parameters |
+--------------------------------------+------------+
|             conv1.weight             |    4032    |
|              conv1.bias              |     64     |
|             conv2.weight             |   36864    |
|              conv2.bias              |     64     |
|      trunk.0.rdb.0.rb.0.weight       |   36864    |
|       trunk.0.rdb.0.rb.0.bias        |     64     |
|      trunk.0.rdb.1.rb.0.weight       |   73728    |
|       trunk.0.rdb.1.rb.0.bias        |     64     |
|      trunk.0.rdb.2.rb.0.weight       |   110592   |
|       trunk.0.rdb.2.rb.0.bias        |     64     |
|      trunk.0.rdb.3.rb.0.weight       |   147456   |
|       trunk.0.rdb.3.rb.0.bias        |     64     |
|      trunk.0.rdb.4.rb.0.weight       |   184320   |
|       trunk.0.rdb.4.rb.0.bias        |     64     |
|      trunk.0.rdb.5.rb.0.weight       |   221184   |
|       trunk.0.rdb.5.rb.0.b

22717440